## Adaptive Multi-Level Student Grade Prediction
### Architecture: K-Means Clustering (Course-based) + Per-Cluster Adaptive Regression

**Pipeline:**
1. K-Means segments courses by credit, subject area, course year, mean grade,
   mean student GPA, grade std-dev, and grade distribution
2. Each student-course instance is assigned to its course's cluster directly
   (no majority voting — one course = one cluster)
3. Per-cluster cross-validated model selection among 9 regression candidates
4. Best model per cluster predicts test semester grades
5. GPA fallback only when regression features are unavailable (~0% of cases)


In [3]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import recommendationsv4 as recommendations

from scipy.spatial.distance import euclidean, cdist

from scipy.spatial.distance import euclidean
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import numpy as np


from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import Ridge, Lasso, LinearRegression, BayesianRidge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import HistGradientBoostingRegressor

#from cluster_model__adaptive_regression__Student_based_KMeans_clean import threshold_values

#from cluster_model__adaptive_regression__Student_based_KMeans_ANHUI import threshold_values

# Minimum cluster size to use local model instead of global fallback.
# Clusters below this threshold use the global training set for CF/regression.


### 1.1 Load regression feature matrix

Full one-hot encoded dataframe used for regression model training and feature extraction.

In [4]:
# Load raw dataset
df_raw = pd.read_csv('../datasets/SEHIR/processed_dataset.csv')

# Build regression-ready encoded dataframe.
# Mirrors preprocessing in the regression notebook so feature spaces match.
df_reg = df_raw.copy()
df_reg = pd.concat([
    df_reg,
    pd.get_dummies(df_reg['Course Year'],     prefix='Course Year'),
    pd.get_dummies(df_reg['Department Code'], prefix='Department Code'),
    pd.get_dummies(df_reg['Course Level'],    prefix='Course Level'),
    pd.get_dummies(df_reg['Standing'],        prefix='Standing'),
    pd.get_dummies(df_reg['Status'],          prefix='Status')
], axis=1)
df_reg.drop(['Course Year', 'Department Code', 'Course Level',
             'Status', 'Standing'], axis=1, inplace=True)

# Label encoder: maps letter grades to integers 0–12 for regression targets.
# A+ → 0, A → 1, ..., F → 12
le_reg = LabelEncoder()
le_reg.fit(['A+', 'A', 'A-', 'B+', 'B', 'B-', 'C+', 'C', 'C-', 'D+', 'D', 'D-', 'F'])

reg_columns = df_reg.columns

# Tuned hyperparameters for each regression model (from separate tuning notebook)
with open('../hyperparameters/tuned_hyperparams (course based).json') as fr:
    tuned_hyperparams_reg = json.load(fr)

print(f"df_reg shape: {df_reg.shape}")

# 1. Create a helper to rank semesters chronologically
def get_semester_rank(sem_name):
    parts = str(sem_name).split(' - ')
    if len(parts) != 2: return 0

    year = int(parts[0])
    season = parts[1].strip().lower()
    season_weights = {'spring': 1, 'summer': 2, 'fall': 3}

    # Creates a unique, sortable integer (e.g., 2011 - Spring -> 20111)
    return year * 10 + season_weights.get(season, 4)


df_reg shape: (48741, 63)


### 1.2 Load clustering / CF matrix

Course-based clustering: each row is a student-course-semester instance.
Clustering features are **course-level** attributes:
- Course Credit, Course Year (one-hot), Subject (one-hot)
- Aggregate statistics computed over the training window:
  Mean Student GPA, Mean Grade, STDEV GPA, STDEV Grade, Grade Distribution

Source file: `processed_course_clustering_dataset.csv`


In [5]:
# Load course-clustering dataset.
# Columns 0-5 are identifiers/metadata; columns 6+ are aggregate course stats
# (Mean GPA, Mean Grade, StdDev GPA, StdDev Grade, grade-distribution buckets).
df = pd.read_csv('../datasets/SEHIR/processed_course_clustering_dataset.csv')

# Keep identifier columns + all aggregate course-stat columns (index 6 onward
# after the slice used in the course-based CF notebook).
df = df[['Student Number', 'Course Code', 'Letter Grade', 'Semester',
         'Course Credit', 'Course Year'] + list(df.columns[20:])]

# One-hot encode Subject and Course Year — these are the two categorical
# course-level features; all other kept columns are already numeric.
df = pd.concat([
    df,
    pd.get_dummies(df['Subject'],      prefix='Subject'),
    pd.get_dummies(df['Course Year'],  prefix='Course Year'),
], axis=1)
df.drop(['Subject', 'Course Year'], axis=1, inplace=True)

print(f"df shape: {df.shape}")
clustering_columns = list(df.columns[5:18]) + list(df.columns[19:20]) + list(df.columns[21:22])
print(f"Clustering feature columns (index 4 onwards): {clustering_columns[:15]} ...")
df.head()

# 2. Sort the dataframe chronologically WITHOUT modifying the dataframe schema
semester_col_name = df.columns[3]

# Create a temporary array of the sorted indices
# We map the get_semester_rank function over the semester column to get the raw ranks
ranks = df[semester_col_name].map(get_semester_rank).values
sorted_indices = np.argsort(ranks)

# 3. Apply the sorted indices to the dataframe and reset the index
df = df.iloc[sorted_indices].reset_index(drop=True)

print("Dataset successfully sorted through time!")

df shape: (48741, 62)
Clustering feature columns (index 4 onwards): ['A+ rate', 'A rate', 'A- rate', 'B+ rate', 'B rate', 'B- rate', 'C+ rate', 'C rate', 'C- rate', 'D+ rate', 'D rate', 'D- rate', 'F rate', 'Mean Grade - Students taken', 'STDEV Grade - Students taken'] ...
Dataset successfully sorted through time!


### 1.3 Grade mappings and course credit lookup

In [6]:
# Numerical grade scale used throughout the pipeline
numerical_grades = {
    'A+': 4.1, 'A': 4.0, 'A-': 3.7,
    'B+': 3.3, 'B': 3.0, 'B-': 2.7,
    'C+': 2.3, 'C': 2.0, 'C-': 1.7,
    'D+': 1.3, 'D': 1.0, 'D-': 0.5,
    'F':  0.0
}

# Course credit lookup: {course_code: credit_hours}
# Used for credit-weighted GPA computation
course_credits = {}
for row_idx in df.index:
    course_code = df.iloc[row_idx, 1]
    credit      = df.iloc[row_idx, 4]
    course_credits[course_code] = credit

## Section 2: Helper Functions

In [7]:
def get_semester_data(semester_name):
    """
    Extract all student-course-grade records for a given semester.

    Returns:
        dict: {student_number: {course_code: numerical_grade, ...}, ...}
    """
    semester_data = {}
    dataset = df[df.iloc[:, 3] == semester_name]
    dataset.index = range(len(dataset))

    for row_idx in dataset.index:
        student_number = dataset.iloc[row_idx, 0]
        course_code    = dataset.iloc[row_idx, 1]
        letter_grade   = dataset.iloc[row_idx, 2]

        semester_data.setdefault(student_number, {})
        semester_data[student_number][course_code] = numerical_grades[letter_grade]

    return semester_data

In [8]:
  # Helper function for Smart Cold-Start Fallback
def get_course_avg(global_data, target_course):
    """Safely calculates the historical average of a specific course across all students."""
    try:
        grades = [courses[target_course] for stu, courses in global_data.items() if target_course in courses]
        return np.mean(grades) if len(grades) > 0 else 2.5 # 2.5 is the absolute cold-start median
    except Exception:
        return 2.5

def get_avg_gpa(train_semester, student):
    """
    Compute credit-weighted GPA for a student from their training history.

    Args:
        train_semester: {student: {course: grade, ...}, ...}
        student: student identifier

    Returns:
        float: weighted GPA
    """
    courses      = train_semester[student]
    total_credit = 0
    weights      = 0
    for course in courses:
        total_credit += course_credits[course]
        weights      += courses[course] * course_credits[course]
    if total_credit > 0:
        return weights / total_credit
    else:
        print("DEBUG: WARNING: Division by zero fallback GPA to 2.5")
        return 2.5


def get_grade_stats(semester_data, student):
    """
    Compute mean and std of a student's grades in the given semester data.
    Used for the outlier filter on pure CF predictions.

    Returns:
        (mean, std_dev): floats
    """
    grade_list = [semester_data[student][course] for course in semester_data[student]]
    return np.mean(grade_list), np.std(grade_list)


def get_course_stats(train_semester):
    """
    Single-pass computation of course means, global grade mean, and mean GPA.
    Called once per semester to avoid redundant iteration.

    Returns:
        course_means (dict): {course_code: mean_grade}
        global_mean  (float): mean grade across all students/courses
        mean_gpa     (float): mean student GPA across all students
    """
    course_grades = {}
    all_grades    = []
    student_gpas  = []

    for student, courses in train_semester.items():
        grades = list(courses.values())
        all_grades.extend(grades)
        if grades:
            student_gpas.append(np.mean(grades))
        for course, grade in courses.items():
            course_grades.setdefault(course, []).append(grade)

    course_means = {c: np.mean(g) for c, g in course_grades.items()}
    global_mean  = np.mean(all_grades)   if all_grades    else 0.0
    mean_gpa     = np.mean(student_gpas) if student_gpas  else 2.0

    return course_means, global_mean, mean_gpa

## Section 3: Regression Helpers

Functions for building training data, candidate model sets, per-cluster model selection, and feature extraction for test instances.

In [9]:
def get_reg_train_data2(train_sems):
    dataFrame = pd.DataFrame(columns=reg_columns)
    for sem in train_sems:
        sem_df = df_reg[df_reg['Semester'] == sem]
        dataFrame = pd.concat([dataFrame, sem_df], ignore_index=True)

    X = dataFrame.drop('Semester', axis=1)
    y = le_reg.transform(X.pop('Letter Grade'))
    X = X.select_dtypes(include=[np.number])
    return X, y

def get_reg_train_data(train_sems):
    # dataFrame = pd.DataFrame(columns=reg_columns)
    # for sem in train_sems:
    #     sem_df = df_reg[df_reg['Semester'] == sem]
    #     #dataFrame = pd.concat([dataFrame, sem_df], ignore_index=True)
    #
    #     # Before pd.concat in your loop
    #     if sem_df is not None and not sem_df.empty:
    #         print(f"DEBUG: Adding {len(sem_df)} rows from {sem}")
    #         dataFrame = pd.concat([dataFrame, sem_df], ignore_index=True)
    #     else:
    #         print(f"DEBUG: WARNING - Semester {sem} returned NO data.")

    # 1. Start with an empty list
    train_list = []

    for sem in train_sems:
        sem_df = df_reg[df_reg['Semester'] == sem]

        if sem_df is not None and not sem_df.empty:
            #print(f"DEBUG: Adding {len(sem_df)} rows from {sem}")
            train_list.append(sem_df)
        else:
            print(f"DEBUG: WARNING - Semester {sem} returned NO data.")

    # 2. Concatenate once at the end
    if train_list:
        dataFrame = pd.concat(train_list, ignore_index=True)
    else:
        # Handle the case where no semesters were found
        dataFrame = pd.DataFrame(columns=reg_columns)

    # 3. Explicit Type Enforcement (The "nan" Fix)
    # Force everything except the identifiers back to numeric
    # to satisfy the new computer's stricter environment
    cols_to_fix = [c for c in dataFrame.columns if c not in ['Semester', 'Student Number', 'Course Code', 'Letter Grade']]
    for col in cols_to_fix:
        dataFrame[col] = pd.to_numeric(dataFrame[col], errors='coerce').fillna(0)

    # 1. Separate target
    y = le_reg.transform(dataFrame.pop('Letter Grade'))

    # 2. Drop non-feature metadata
    X = dataFrame.drop(['Semester', 'Student Number', 'Course Code', 'Course Title'],
                       axis=1, errors='coerce')

    # 1. Systematically convert all potential feature columns to numeric
    for col in dataFrame.columns:
        # errors='coerce' ensures that true categorical strings stay as objects
        # while numeric strings like "33.0" become floats
        dataFrame[col] = pd.to_numeric(dataFrame[col], errors='coerce')

    # 3. FIX: Include booleans or convert them to integers
    # This ensures one-hot encoded columns aren't deleted
    # 3. Now the selector will catch EVERYTHING that is numeric or boolean
    #X = dataFrame.select_dtypes(include=[np.number, 'bool']).astype(float)



    # Drop remaining metadata from X
    #X = X.drop([c for c in metadata_cols if c in X.columns], axis=1)

    return X, y

def get_student_course_features(student_number, course_code, semester, sc):
    """
    Extract and scale the regression feature vector for a single
    student-course-semester triple from df_reg.

    Aligns columns to exactly what the StandardScaler saw during fit,
    filling missing one-hot columns with 0.

    Returns:
        scaled numpy array (1 x n_features), or None if the row is not found
    """
    mask = ((df_reg['Student Number'] == student_number) &
            (df_reg['Course Code']    == course_code) &
            (df_reg['Semester']       == semester))
    rows = df_reg[mask]
    if len(rows) == 0:
        return None

    row = rows.iloc[0].copy()

    # Drop non-feature columns
    for col in ['Semester', 'Letter Grade', 'Student Number', 'Course Code']:
        if col in row.index:
            row = row.drop(col)

    # Align to scaler's expected column order; fill unseen one-hot columns with 0
    row = row[row.index.isin(sc.feature_names_in_)]
    aligned = pd.Series(0.0, index=sc.feature_names_in_)
    aligned.update(row)

    features_df = pd.DataFrame([aligned], columns=sc.feature_names_in_)
    try:
        return sc.transform(features_df)
    except Exception:
        return None

In [1]:
import warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.linear_model import Ridge, Lasso, LinearRegression, BayesianRidge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

import warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.linear_model import Ridge, Lasso, LinearRegression, BayesianRidge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import BaggingRegressor

def fit_regression_models_per_cluster(train_sems, cluster_labels, train_dataset, tuned_hyperparams, threshold,
                                      cluster_raw_data=None, sim_func=None, test_all_models=False,
                                      train_semester_global=None,
                                      model_selection_only=False):

    X_train, y_train = get_reg_train_data(train_sems)

    if len(X_train) == 0:
        return {}, None, {}, {}

    sc = StandardScaler()
    sc.fit(X_train)

    cluster_X = {}
    cluster_y = {}
    cluster_chron = {}
    cluster_sem = {}

    sem_col = 'Semester' if 'Semester' in train_dataset.columns else train_dataset.columns[3]

    train_vals = train_dataset.values
    X_train_vals = X_train.values
    y_train_vals = y_train.values if hasattr(y_train, 'values') else np.array(y_train)
    sem_col_idx = train_dataset.columns.get_loc(sem_col)

    for i in range(len(train_vals)):
        sn = train_vals[i, 0]
        cc = train_vals[i, 1]
        sem = train_vals[i, sem_col_idx]
        lbl = cluster_labels.get((sn, cc))

        if lbl is not None and i < len(X_train_vals):
            cluster_X.setdefault(lbl, []).append(X_train_vals[i])
            y_val = float(y_train_vals[i])
            cluster_y.setdefault(lbl, []).append(y_val)
            cluster_chron.setdefault(lbl, []).append((sn, cc, y_val))
            cluster_sem.setdefault(lbl, []).append(sem)

    def get_full_candidates2():
        return {
            #'Ridge':         Ridge(alpha=tuned_hyperparams['Ridge']['alpha']),
            #'Lasso':         Lasso(alpha=tuned_hyperparams['Lasso']['alpha']),
            #'LR':            LinearRegression(),
            #'KNN':           KNeighborsRegressor(),
            'SVR':           SVR(C=10, kernel='rbf', epsilon=0.5, gamma=0.1),
            #'RF':            RandomForestRegressor(n_estimators=tuned_hyperparams['RandomForestRegressor']['n_estimators'], random_state=42, n_jobs=-1),
            #'GBM':           HistGradientBoostingRegressor(                                learning_rate=tuned_hyperparams['GradientBoostingRegressor']['learning_rate'],max_depth=tuned_hyperparams['GradientBoostingRegressor']['max_depth'],random_state=42),
            #'AdaBoost':      AdaBoostRegressor(n_estimators=tuned_hyperparams['AdaBoostRegressor']['n_estimators'], learning_rate=tuned_hyperparams['AdaBoostRegressor']['learning_rate'], random_state=42),
            'MLP':           MLPRegressor(hidden_layer_sizes=(128, 256, 64), activation='relu', solver='adam', max_iter=500, random_state=42)  ## 500, 200
            #'BayesianRidge': BayesianRidge(lambda_1=tuned_hyperparams['BayesianRidge']['lambda_1'], lambda_2=tuned_hyperparams['BayesianRidge']['lambda_2'], alpha_1=tuned_hyperparams['BayesianRidge']['alpha_1'], alpha_2=tuned_hyperparams['BayesianRidge']['alpha_2']),
        }

    def get_full_candidates():
        """
        Returns a dictionary of regression models with explicit parameters
        optimized for student grade prediction and course-based clusters.
        """
        return {
            # Linear models with L2/L1 regularization to handle correlated course features
            'Ridge':         Ridge(alpha=1000),
            'Lasso':         Lasso(alpha=1),
            #'LR':            LinearRegression(),
            #'BayesianRidge': BayesianRidge(alpha_1=1e-6, lambda_1=1e-6),

            # Local geometry models: 'distance' weighting helps for students with similar histories
            'KNN':           KNeighborsRegressor(n_neighbors=5, weights='distance'),

            # SVR handles small-cluster non-linearity well; epsilon=0.1 ignores minor noise
            'SVR':           SVR(C=10.0, kernel='rbf', epsilon=0.1, gamma='scale'),

            # Ensemble methods with guards against overfitting (max_depth and min_samples)
            'RF':            RandomForestRegressor(
                                n_estimators=400,
                                max_depth=12,
                                min_samples_leaf=8,
                                max_features='sqrt',
                                n_jobs=-1,
                                random_state=42
                             ),

            'GBM': HistGradientBoostingRegressor(
                        learning_rate=0.05,        # Slow down learning
                        max_depth=4,               # Keep trees shallow
                        l2_regularization=0.5,     # Add penalty
                        max_iter=1000,             # Allow many trees...
                        n_iter_no_change=15,       # ...but stop if validation doesn't improve for 15 rounds
                        validation_fraction=0.1,   # Use 10% of cluster for internal validation
                        random_state=42
                    ),

            # 'AdaBoost':      AdaBoostRegressor(
            #                     n_estimators=50,
            #                     learning_rate=0.1,
            #                     random_state=42
            #                  ),

            # The "Bagging" method from the paper
            'Bagging':       BaggingRegressor(
                                n_estimators=1000,
                                max_samples=0.8, # 80% bootstrap for diversity
                                random_state=42
                             ),

            # Neural Network: Tapered architecture with early stopping to prevent memorization
            'MLP':           MLPRegressor(
                                hidden_layer_sizes=(128, 64, 32),
                                activation='relu',
                                solver='adam',
                                alpha=0.01,         # L2 regularization
                                max_iter=1000,
                                random_state=42
                             )
        }

    def get_full_candidates0():
        return {
            #'Ridge':         Ridge(alpha=tuned_hyperparams['Ridge']['alpha']),
            'Lasso':         Lasso(alpha=tuned_hyperparams['Lasso']['alpha']),
            #'LR':            LinearRegression(),
            'KNN':           KNeighborsRegressor(),
            'SVR':           SVR(C=10, kernel='rbf', epsilon=0.5, gamma=0.1),
            'RF':            RandomForestRegressor(n_estimators=tuned_hyperparams['RandomForestRegressor']['n_estimators'], random_state=42, n_jobs=-1),
            'GBM':           HistGradientBoostingRegressor(
                                learning_rate=tuned_hyperparams['GradientBoostingRegressor']['learning_rate'],
                                max_depth=tuned_hyperparams['GradientBoostingRegressor']['max_depth'],
                                random_state=42
                             ),
            #'AdaBoost':      AdaBoostRegressor(n_estimators=tuned_hyperparams['AdaBoostRegressor']['n_estimators'], learning_rate=tuned_hyperparams['AdaBoostRegressor']['learning_rate'], random_state=42),
            'MLP': MLPRegressor(hidden_layer_sizes=(200, 100), activation='relu', solver='adam', max_iter=100, random_state=42),  ## 500, 200 ## (400, 200, 100) ## 200, 400, 100
           'Bagging':       BaggingRegressor(n_estimators=50, random_state=42)
            #'BayesianRidge': BayesianRidge(lambda_1=tuned_hyperparams['BayesianRidge']['lambda_1'], lambda_2=tuned_hyperparams['BayesianRidge']['lambda_2'], alpha_1=tuned_hyperparams['BayesianRidge']['alpha_1'], alpha_2=tuned_hyperparams['BayesianRidge']['alpha_2']),
        }

    def _evaluate_cf_chronological(t_inst, v_inst):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            if len(t_inst) < 5 or len(v_inst) == 0: return float('inf')

            tr_matrix = {}
            for s, c, g in t_inst: tr_matrix.setdefault(s, {})[c] = g
            f_sims = recommendations.calculateSimilarItems(tr_matrix)
            g_loc = {s: get_avg_gpa(tr_matrix, s) for s in tr_matrix}

            yt, yp = [], []

            for s, c, g in v_inst:
                yt.append(g)
                fp = None
                rec = {}
                try:
                    for rg, rc in recommendations.getRecommendedItems(tr_matrix, f_sims, s): rec[rc] = rg
                except: pass
                try:
                    for rg, rc in recommendations.getRecommendations(tr_matrix, s, sim_func, dgpa=True, gpa=g_loc, delta=0.7):
                        if rc not in rec: rec[rc] = rg
                except: pass

                if c in rec: fp = rec[c]
                else:
                    try: fp = g_loc.get(s, None)
                    except: fp = None

                if fp is None or np.isinf(fp) or np.isnan(fp):
                    stu_hist = [g_ for s_, c_, g_ in t_inst if s_ == s]
                    fp = np.mean(stu_hist) if len(stu_hist) > 0 else None

                if fp is None or np.isinf(fp) or np.isnan(fp):
                    crs_hist = [g_ for s_, c_, g_ in t_inst if c_ == c]
                    fp = np.mean(crs_hist) if len(crs_hist) > 0 else 2.5

                yp.append(fp)

            return np.sqrt(mean_squared_error(yt, yp))

    reg_models        = {}
    cv_scores_log     = {}
    all_fitted_models = {}

    debug = True

    for lbl in cluster_X:
        print(f'Cluster: {lbl}')
        Xc_raw = np.array(cluster_X[lbl])
        yc = np.array(cluster_y[lbl])
        semc = np.array(cluster_sem[lbl])
        chronc = cluster_chron[lbl]

        is_chronological = len(train_sems) > 1

        # ─── DUAL-TARGET TEMPORAL SETUP ───
        start_ix = 0
        if is_chronological:
            validation_targets = [train_sems[-2]]
            if 'Spring' in validation_targets[0]:
                start_ix = 0
            else:
                start_ix = 1
        else:
            validation_targets = [train_sems[-1]]

        # if is_chronological:
        #     if len(train_sems) % 2 == 0:
        #         validation_targets = [train_sems[1]]
        #     else:
        #         validation_targets = [train_sems[0]]
        # else:
        #     validation_targets = [train_sems[0]]


        print(f'Validation target: {validation_targets[0]}')

        if len(Xc_raw) < 5:
            cv_scores_log[lbl] = {'winner': None, 'n': len(yc), 'winner_score': float('inf'), 'all_scores': {}}
            continue

        best_name   = None
        best_score  = np.inf
        all_scores  = {}

        candidates_to_test = get_full_candidates()

        # 1. Force both to clean, stripped strings
        semc_clean = np.array([str(x).strip() for x in semc])

        EightyTwenty_Validation = False
        for model_selection_attempt in range(2):
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=RuntimeWarning)
                warnings.simplefilter("ignore", category=UserWarning)

                debug = True
                for name, m in candidates_to_test.items():
                    #try:
                        #print(f'  Training model: {name}')
                        target_rmses = []

                        for target_sem in validation_targets:
                            val_sem_clean = str(target_sem).strip()

                            #2. Robust Slicing: Only use Even-Odd if you have enough history (e.g., > 2 semesters)
                            if len(train_sems) > 2:
                                alternating_sems = [str(s).strip() for s in train_sems[start_ix::2] if val_sem_clean != str(s).strip()]
                            else:
                                # Fallback: Use all historical semesters that aren't the validation target
                                alternating_sems = [str(s).strip() for s in train_sems if str(s).strip() != val_sem_clean]

                            if len(alternating_sems) == 0:
                                alternating_sems = [str(s).strip() for s in [train_sems[-1]]]

                            #alternating_sems = [str(s).strip() for s in train_sems if str(s).strip() != val_sem_clean]

                            # if len(train_sems) > 2:
                            #     alternating_sems = [str(s).strip() for s in train_sems[-4:-1]]
                            # elif len(train_sems) > 1:
                            #     alternating_sems = [str(s).strip() for s in train_sems[-2:-1]]
                            # else:
                            #     alternating_sems = [str(s).strip() for s in train_sems[-1]]

                            # 3. Apply Mask
                            mask_train = np.isin(semc_clean, alternating_sems)
                            mask_val = semc_clean == val_sem_clean

                            ##############################
                            #use 80% of whole training data
                            # alternating_sems = [str(s).strip() for s in train_sems]
                            #
                            # mask_train = np.isin(semc_clean, alternating_sems)
                            # mask_val = mask_train

                            #######################
                            if EightyTwenty_Validation:
                                if debug:
                                    debug = False
                                    print('80 - 20 case (FALLBACK TRIGGERED: Pooling all historical semesters)...')

                                # Combine all historical train semesters and target semesters for this cluster
                                allowed_sems = [str(s).strip() for s in train_sems] + [val_sem_clean]
                                mask_combined = np.isin(semc_clean, allowed_sems)

                                idx_combined = np.where(mask_combined)[0]
                                if len(idx_combined) < 5:
                                    continue

                                # Shuffling or deterministic split over the pooled semesters
                                state = np.random.RandomState(42)
                                shuffled_indices = state.permutation(len(idx_combined))
                                split_idx = int(len(idx_combined) * 0.8)

                                train_idx = idx_combined[shuffled_indices[:split_idx]]
                                val_idx = idx_combined[shuffled_indices[split_idx:]]

                                X_tr_fold_raw = Xc_raw[train_idx]
                                y_tr_fold = yc[train_idx]
                                X_val_fold_raw = Xc_raw[val_idx]
                                y_val_fold = yc[val_idx]

                            # ── FLIGHT SAFETY FALLBACK: REGRESSION ──
                            # If masking the target semester empties the training data, we dynamically split the target
                            elif not is_chronological or len(train_sems) < 2:
                                idx_target = np.where(mask_val)[0]
                                #if len(idx_target) < 5:
                                #    continue # Cannot run on micro-data

                                split_idx = int(len(idx_target) * 0.8)
                                train_idx = idx_target[:split_idx]
                                val_idx = idx_target[split_idx:]

                                X_tr_fold_raw = Xc_raw[train_idx]
                                y_tr_fold = yc[train_idx]
                                X_val_fold_raw = Xc_raw[val_idx]
                                y_val_fold = yc[val_idx]

                                if debug:
                                    debug = False
                                    print('80 - 20 case...')
                                    print(f"  DEBUG: Alternating sems: {alternating_sems}")
                                    print(f"  DEBUG: Training rows: {np.sum(mask_train)}")

                                #print(f'    total data: {len(idx_target)} train: {len(train_idx)} test: {len(val_idx)}')
                            else:

                                if debug:
                                    debug = False
                                    print(f"   DEBUG: Alternating sems: {alternating_sems}")
                                    print(f"   DEBUG: Training rows: {np.sum(mask_train)}")

                                X_tr_fold_raw = Xc_raw[mask_train]

                                y_tr_fold = yc[mask_train]
                                X_val_fold_raw = Xc_raw[mask_val]
                                y_val_fold = yc[mask_val]

                            if len(X_tr_fold_raw) < 5 or len(X_val_fold_raw) == 0:
                                continue

                            local_sc_fold = StandardScaler()
                            X_tr_fold = local_sc_fold.fit_transform(X_tr_fold_raw)
                            X_val_fold = local_sc_fold.transform(X_val_fold_raw)

                            m.fit(X_tr_fold, y_tr_fold)
                            preds = m.predict(X_val_fold)

                            target_rmses.append(np.sqrt(mean_squared_error(y_val_fold, preds)))

                        if len(target_rmses) > 0:
                            final_candidate_score = np.mean(target_rmses)
                            all_scores[name] = round(final_candidate_score, 5)

                            if model_selection_only:
                                print(f"  {name}: {final_candidate_score}")

                            if final_candidate_score < best_score:
                                best_score = final_candidate_score
                                best_name  = name
                        else:
                            all_scores[name] = None

                    # except Exception:
                    #     all_scores[name] = None
                    #     continue

                if len(Xc_raw) >= threshold and sim_func is not None and train_semester_global is not None:
                    try:
                        cf_target_rmses = []
                        for target_sem in validation_targets:
                            mask_val = (semc == target_sem)
                            mask_train = (semc != target_sem) & np.isin(semc, train_sems)

                            t_inst = [chronc[j] for j in range(len(chronc)) if mask_train[j]]

                            # ── FLIGHT SAFETY FALLBACK: COLLABORATIVE FILTERING ──
                            if len(t_inst) < 5:
                                idx_target = np.where(mask_val)[0]
                                if len(idx_target) < 5:
                                    continue

                                split_idx = int(len(idx_target) * 0.8)
                                train_idx = idx_target[:split_idx]
                                val_idx = idx_target[split_idx:]

                                t_inst = [chronc[j] for j in train_idx]
                                v_inst = [chronc[j] for j in val_idx]
                            else:
                                v_inst = [chronc[j] for j in range(len(chronc)) if mask_val[j]]

                            if len(t_inst) < 5 or len(v_inst) == 0:
                                continue

                            cf_rmse = _evaluate_cf_chronological(t_inst, v_inst)
                            if not np.isinf(cf_rmse):
                                cf_target_rmses.append(cf_rmse)

                        if len(cf_target_rmses) > 0:
                            cf_final_score = np.mean(cf_target_rmses)
                            all_scores['CF'] = round(cf_final_score, 5)

                            if cf_final_score < best_score:
                                best_name  = 'CF'
                                best_score = cf_final_score
                        else:
                            all_scores['CF'] = None
                    except Exception:
                        all_scores['CF'] = None

            # If we found a valid model, break out of the 2-pass loop immediately
            if best_name is not None:
                break

            # If everything failed, flip the switch and restart the loop
            if is_chronological and len(train_sems) >= 2 and not EightyTwenty_Validation:
                print(f"  [DEBUG FALLBACK] Cluster {lbl} returned 'None'. Restarting validation with 80/20 data split...")
                EightyTwenty_Validation = True
            else:
                break # Even fallback failed (or fallback wasn't applicable)

        print(f"  Cluster {lbl}: winner = {best_name} (n={len(Xc_raw)}, AvgRMSE={best_score:.4f})")

        if not model_selection_only:
            # ─── FINAL RETRAINING ───
                    # 1. Merge the data
            X_combined_raw = np.concatenate([X_tr_fold_raw, X_val_fold_raw], axis=0)
            y_combined = np.concatenate([y_tr_fold, y_val_fold], axis=0)

            # 2. Scale the combined data
            # Use a fresh scaler fit on the combined set to avoid leakage or skew
            if len(X_combined_raw) > 0:
                final_scaler = StandardScaler()
                X_combined_scaled = final_scaler.fit_transform(X_combined_raw)
                X_combined_scaled = np.nan_to_num(X_combined_scaled) # Safety guard
            else:
                Xcombined_scaled = X_combined_raw

            if best_name == 'CF':
                reg_models[lbl] = 'CF'
            elif best_name is not None:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", category=UserWarning)
                    Xc_scaled = sc.transform(Xc_raw)

                winner_model = get_full_candidates()[best_name]
                #winner_model.fit(Xc_scaled, yc)
                winner_model.fit(X_combined_scaled, y_combined)
                reg_models[lbl] = winner_model

            if test_all_models:
                all_fitted_models[lbl] = {}
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", category=UserWarning)
                    Xc_scaled = sc.transform(Xc_raw)

                for name, m in get_full_candidates().items():
                    if best_name == name:
                        all_fitted_models[lbl][name] = winner_model
                        continue
                    try:
                        #m.fit(Xc_scaled, yc)
                        m.fit(X_combined_scaled, y_combined)
                        all_fitted_models[lbl][name] = m
                    except Exception:
                        pass
                if lbl in cluster_chron:
                    all_fitted_models[lbl]['CF'] = True

        cv_scores_log[lbl] = {
            'winner':       best_name,
            'n':            len(yc),
            'winner_score': best_score,
            'all_scores':   all_scores,
        }

    return reg_models, sc, cv_scores_log, all_fitted_models

In [11]:

def evaluate_cf_cv(cluster_data, sim, train_semester_global):
    """Fair, Leak-Proof 5-Fold CV for CF."""
    instances = []
    for s, courses in cluster_data.items():
        for c, g in courses.items():
            instances.append((s, c, g))

    if len(instances) < 5:
        return float('inf')

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    fold_rmses = []

    for train_idx, test_idx in kf.split(instances):
        train_inst = [instances[i] for i in train_idx]
        test_inst = [instances[i] for i in test_idx]

        train_data_fold = {}
        for s, c, g in train_inst:
            train_data_fold.setdefault(s, {})[c] = g

        fold_sims = recommendations.calculateSimilarItems(train_data_fold)
        gpa_local = {s: get_avg_gpa(train_data_fold, s) for s in train_data_fold}

        # --- PREVENT DATA LEAKS ---
        # Track which students are in the test fold for each course
        exclude_s_for_c = {}
        for s, c, g in test_inst:
            exclude_s_for_c.setdefault(c, set()).add(s)

        def safe_student_gpa(stu, crs_to_exclude):
            """Calculates Global Student GPA without peeking at the target course grade."""
            hist = train_semester_global.get(stu, {})
            valid = [val for crs, val in hist.items() if crs != crs_to_exclude]
            return np.mean(valid) if valid else None

        def safe_course_avg(crs, stu_to_exclude):
            """Calculates Global Course Average without peeking at the test students' grades."""
            valid = [courses[crs] for stu, courses in train_semester_global.items()
                     if crs in courses and stu not in stu_to_exclude]
            return np.mean(valid) if valid else 2.5
        # --------------------------

        y_true = []
        y_pred = []

        for s, c, g in test_inst:
            y_true.append(g)
            final_pred = None

            # 1. Try CF Prediction
            recommended = {}
            try:
                for rg, rc in recommendations.getRecommendedItems(train_data_fold, fold_sims, s):
                    recommended[rc] = rg
            except: pass
            try:
                for rg, rc in recommendations.getRecommendations(train_data_fold, s, sim, dgpa=True, gpa=gpa_local, delta=0.7):
                    if rc not in recommended:
                        recommended[rc] = rg
            except: pass

            # --- THE FAIR, LEAK-PROOF FALLBACK CASCADE ---
            if c in recommended:
                final_pred = recommended[c]
            else:
                try: final_pred = gpa_local[s] # Fallback 1: Local Cluster GPA (safe, uses 80% split)
                except KeyError: final_pred = None

            if final_pred is None or np.isinf(final_pred) or np.isnan(final_pred):
                final_pred = safe_student_gpa(s, c) # Fallback 2: Leak-Proof Global Student GPA

            if final_pred is None or np.isinf(final_pred) or np.isnan(final_pred):
                final_pred = safe_course_avg(c, exclude_s_for_c.get(c, set())) # Fallback 3: Leak-Proof Global Course Avg

            y_pred.append(final_pred)

        fold_rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        fold_rmses.append(fold_rmse)

    return np.mean(fold_rmses)

## Section 4: Clustering Functions

In [12]:
def elbow_inertia(train_sems, cluster_model, k_range=range(10, 31, 5)):
    """
    Compute K-Means inertia for a range of k values using the elbow method.

    Clustering features: all df columns from index 4 onwards (course-level
    attributes: credit, one-hot subject/year, aggregate grade statistics).
    """
    train_dataset = pd.DataFrame(columns=df.columns)
    for sem in train_sems:
        train_dataset = pd.concat(
            [train_dataset, df[df.iloc[:, 3] == sem]], ignore_index=True)

    # Course-based features start at column 4
    cluster_features = train_dataset[clustering_columns]

    # Before scaling features
    variances = df[clustering_columns].var()
    zero_var_cols = variances[variances == 0].index.tolist()

    if zero_var_cols:
        print(f"DEBUG: Found {len(zero_var_cols)} zero-variance columns: {zero_var_cols}")
        # Option: Fill with a tiny epsilon to avoid division by zero
        for col in zero_var_cols:
            df[col] = df[col] + 1e-9

    # Z-score normalise so numeric range differences don't dominate distances
    from sklearn.preprocessing import StandardScaler as _SS
    cluster_features = pd.DataFrame(
        _SS().fit_transform(cluster_features), columns=cluster_features.columns)

    inertias = []
    for k in k_range:
        kmeans = cluster_model(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(cluster_features)
        inertias.append(kmeans.inertia_)

    return list(k_range), inertias


In [13]:
def merge_small_clusters(cluster_dataset, cluster_label_map,
                          fitted_cluster_model, threshold):
    """
    Iteratively merge clusters below the instance-count threshold into the
    nearest cluster by centroid distance, starting from the smallest.

    For course-based clustering the unit of measurement is student-course
    *instances* (rows), not unique students.  A cluster's size is therefore
    the total number of (student, course) pairs it contains.

    Args:
        cluster_dataset:      {cluster_label: {student: {course: grade}}}
        cluster_label_map:    {(student_number, course_code): cluster_label}
        fitted_cluster_model: fitted KMeans (provides cluster_centers_)
        threshold:            minimum instance count per cluster

    Returns:
        dataset   (dict): merged cluster_dataset
        label_map (dict): updated cluster_label_map
        merge_map (dict): {original_label: final_label}
    """
    centers   = fitted_cluster_model.cluster_centers_
    dataset   = {lbl: {sn: dict(courses)
                       for sn, courses in students.items()}
                 for lbl, students in cluster_dataset.items()}
    label_map = dict(cluster_label_map)
    merge_map = {lbl: lbl for lbl in range(len(centers))}

    def instance_count(cluster_students):
        """Total number of (student, course) pairs in a cluster."""
        return sum(len(courses) for courses in cluster_students.values())

    while True:
        small = [(lbl, instance_count(students))
                 for lbl, students in dataset.items()
                 if instance_count(students) < threshold]
        if not small:
            break

        small.sort(key=lambda x: x[1])
        src_lbl, src_size = small[0]

        src_center = centers[src_lbl]
        best_lbl   = min(
            (lbl for lbl in dataset if lbl != src_lbl),
            key=lambda lbl: np.linalg.norm(src_center - centers[lbl]),
            default=None)

        if best_lbl is None:
            break

        # print(f"  Merging cluster {src_lbl} "
        #       f"(n_instances={src_size}) → cluster {best_lbl} "
        #       f"(n_instances={instance_count(dataset[best_lbl])})")

        # Merge student dicts: if a student appears in both clusters,
        # their courses are combined under the destination cluster.
        for sn, courses in dataset[src_lbl].items():
            dataset[best_lbl].setdefault(sn, {}).update(courses)
        del dataset[src_lbl]

        for orig in merge_map:
            if merge_map[orig] == src_lbl:
                merge_map[orig] = best_lbl

        # Update instance-level label map
        for key in label_map:
            if label_map[key] == src_lbl:
                label_map[key] = best_lbl

    surviving = sorted(dataset.keys())
    print(f"  After merging {len(cluster_dataset)}: {len(surviving)} clusters, "
          f"instance counts: "
          f"{[instance_count(dataset[l]) for l in surviving]}")

    return dataset, label_map, merge_map

import numpy as np
from scipy.spatial.distance import euclidean

def merge_high_inertia_clusters(X, labels, inertia_threshold):
    """
    Evaluates the Mean Inertia (density) of each cluster.
    If a cluster's inertia exceeds the threshold, it merges it with the
    closest neighboring cluster that also exceeds the threshold.

    X: numpy array of feature vectors (n_samples, n_features)
    labels: numpy array of current cluster assignments (n_samples,)
    inertia_threshold: float, the maximum allowed mean squared distance
    """
    return labels


    unique_labels = np.unique(labels)
    centroids = {}
    inertias = {}

    # 1. Calculate Centroids and Mean Inertia for all clusters
    for lbl in unique_labels:
        cluster_points = X[labels == lbl]
        if len(cluster_points) == 0:
            continue

        centroid = np.mean(cluster_points, axis=0)
        centroids[lbl] = centroid

        # Mean Inertia: Average squared Euclidean distance to the centroid
        mean_inertia = np.mean(np.sum((cluster_points - centroid)**2, axis=1))
        inertias[lbl] = mean_inertia

    # 2. Identify the "Loose" clusters
    loose_clusters = set([lbl for lbl, iner in inertias.items() if iner > inertia_threshold])

    # If there is 1 or 0 loose clusters, no pairwise merging can occur
    if len(loose_clusters) < 2:
        return labels

    print(f"       -> [Inertia Merge] Found {len(loose_clusters)} loose clusters exceeding threshold {inertia_threshold}.")

    # 3. Create a merge map to track redirections
    merge_map = {lbl: lbl for lbl in unique_labels}
    active_loose = list(loose_clusters)

    # 4. Pair and merge the loose clusters based on centroid proximity
    while len(active_loose) > 1:
        # Pop the first loose cluster
        c1 = active_loose.pop(0)

        closest_dist = float('inf')
        best_c2 = None

        # Find the closest OTHER loose cluster
        for c2 in active_loose:
            dist = euclidean(centroids[c1], centroids[c2])
            if dist < closest_dist:
                closest_dist = dist
                best_c2 = c2

        if best_c2 is not None:
            merge_map[c1] = best_c2
            # We map c1 into best_c2. best_c2 remains in the active list
            # and acts as the "gravity well" that can absorb other loose clusters.

    # 5. Apply the merge map to the original labels
    final_labels = np.copy(labels)
    for i in range(len(final_labels)):
        curr = final_labels[i]
        # Resolve any merge chains (e.g., if A merged to B, and B merged to C)
        while merge_map[curr] != curr:
            curr = merge_map[curr]
        final_labels[i] = curr

    merged_count = len(np.unique(labels)) - len(np.unique(final_labels))
    if merged_count > 0:
        print(f"       -> [Inertia Merge] Successfully collapsed {merged_count} loose clusters together.")

    return final_labels


In [14]:
def drop_high_inertia_outliers(X_scaled, labels, z_threshold=2.0):
    """
    Identifies and drops individual data points that are statistically
    too far from their cluster's centroid (high inertia contributors).

    Args:
        X_scaled: The standardized feature matrix.
        labels: The current cluster assignments for each row.
        z_threshold: Number of standard deviations above the mean distance
                     to tolerate before dropping a point.
    Returns:
        new_labels: Updated array where outliers are assigned label -1.
    """
    return labels

    new_labels = np.copy(labels)
    unique_labels = np.unique(labels)

    total_dropped = 0

    for lbl in unique_labels:
        if lbl == -1: continue # Skip already dropped points

        # Find indices of points in this cluster
        idx = np.where(labels == lbl)[0]

        # Do not prune extremely small clusters to avoid breaking CV folds
        if len(idx) < 15:
            continue

        cluster_points = X_scaled[idx]
        centroid = np.mean(cluster_points, axis=0)

        # Calculate squared Euclidean distance (inertia contribution)
        sq_distances = np.sum((cluster_points - centroid)**2, axis=1)

        # Calculate cluster-specific topographical threshold
        mean_dist = np.mean(sq_distances)
        std_dist = np.std(sq_distances)
        threshold = mean_dist + (z_threshold * std_dist)

        # Find outliers exceeding the threshold
        outlier_mask = sq_distances > threshold
        outlier_indices = idx[outlier_mask]

        # Mark as dropped (-1)
        if len(outlier_indices) > 0:
            new_labels[outlier_indices] = -1
            total_dropped += len(outlier_indices)

    if total_dropped > 0:
        print(f"       -> [Inertia Pruning] Purged {total_dropped} high-variance outlier instances from regressors.")

    return new_labels

In [15]:
def fit_cluster(train_sems, num_clusters, training_data,
                cluster_model, threshold):
    """
    Fit K-Means on course-level features and assign each student-course
    instance directly to a cluster, with multi-stage structural merging
    and intra-cluster outlier pruning.
    """
    # 1. Initialize an empty list, NOT an empty DataFrame
    train_list = []

    for sem in train_sems:
        # 2. Extract and copy the raw semester data
        sem_df = df[df.iloc[:, 3] == sem].copy()

        # 3. Append only if not empty
        if not sem_df.empty:
            # Optional: Force numeric conversion here to prevent dtype pollution
            # sem_df[clustering_columns] = sem_df[clustering_columns].apply(pd.to_numeric, errors='coerce')
            train_list.append(sem_df)
        else:
            print(f"DEBUG: Warning - No data found for semester: {sem}")

    # 4. Concatenate everything at once
    if train_list:
        train_dataset = pd.concat(train_list, ignore_index=True)
    else:
        # Fallback to an empty DataFrame with correct columns if no data found
        train_dataset = pd.DataFrame(columns=df.columns)

    # 5. FINAL SAFETY: Ensure no 'object' types remain in numeric columns
    # This is crucial for the new machine's environment to avoid 'nan' scores
    for col in clustering_columns:
        if col in train_dataset.columns:
            train_dataset[col] = pd.to_numeric(train_dataset[col], errors='coerce').fillna(0)


    cluster_feature_cols = clustering_columns
    cluster_features_raw = train_dataset[cluster_feature_cols].copy()

    sc_cluster = StandardScaler()
    cluster_features = pd.DataFrame(
        sc_cluster.fit_transform(cluster_features_raw),
        columns=cluster_feature_cols)

    fitted_cluster_model = cluster_model(
        n_clusters=num_clusters, random_state=42).fit(cluster_features)
    cluster_labels = fitted_cluster_model.labels_

    cluster_dataset   = {}
    cluster_label_map = {}

    for i in range(len(cluster_labels)):
        sn  = train_dataset.iloc[i, 0]
        cc  = train_dataset.iloc[i, 1]
        lbl = int(cluster_labels[i])

        cluster_dataset.setdefault(lbl, {})
        cluster_dataset[lbl].setdefault(sn, {})
        if sn in training_data and cc in training_data[sn]:
            cluster_dataset[lbl][sn][cc] = training_data[sn][cc]
        cluster_label_map[(sn, cc)] = lbl

    # ─── 1. MINIMUM SIZE MERGE ────────────────────────────────────────
    cluster_dataset, cluster_label_map, merge_map = merge_small_clusters(
        cluster_dataset, cluster_label_map, fitted_cluster_model,
        threshold=threshold)

    # ─── 2. HIGH-INERTIA MERGE (THE CHAOS COLLAPSER) ──────────────────
    MAX_INERTIA = 4500.0

    size_merged_labels = np.zeros(len(cluster_features), dtype=int)
    for i in range(len(cluster_features)):
        sn  = train_dataset.iloc[i, 0]
        cc  = train_dataset.iloc[i, 1]
        size_merged_labels[i] = cluster_label_map[(sn, cc)]

    final_labels = merge_high_inertia_clusters(cluster_features.values, size_merged_labels, MAX_INERTIA)

    # ─── 3. INTRA-CLUSTER INERTIA PRUNING (THE OUTLIER PURGE) ─────────
    # Drop points that are > 2.0 standard deviations away from their local cluster mean
    Z_THRESHOLD = 100.0
    final_labels = drop_high_inertia_outliers(cluster_features.values, final_labels, z_threshold=Z_THRESHOLD)

    # ─── 4. REBUILD CLUSTER DICTIONARIES ──────────────────────────────
    # cluster_dataset = {}
    # inertia_merge_map = {}
    #
    # # Safely extract the cluster mapping bypassing the -1 outliers
    # for i in range(len(final_labels)):
    #     if final_labels[i] != -1:
    #         inertia_merge_map[size_merged_labels[i]] = int(final_labels[i])
    #
    # # Rebuild dictionaries ONLY for non-outlier instances
    # cluster_label_map = {}
    # for i in range(len(final_labels)):
    #     sn  = train_dataset.iloc[i, 0]
    #     cc  = train_dataset.iloc[i, 1]
    #     new_lbl = int(final_labels[i])
    #
    #     if new_lbl == -1:
    #         continue # Ghost this instance completely; regressors will never see it.
    #
    #     cluster_dataset.setdefault(new_lbl, {})
    #     cluster_dataset[new_lbl].setdefault(sn, {})
    #     if sn in training_data and cc in training_data[sn]:
    #         cluster_dataset[new_lbl][sn][cc] = training_data[sn][cc]
    #
    #     cluster_label_map[(sn, cc)] = new_lbl
    #
    # for original_label, size_merged_label in merge_map.items():
    #     merge_map[original_label] = inertia_merge_map.get(size_merged_label, size_merged_label)

    # ─── 5. RECALCULATE SILHOUETTE SCORE ──────────────────────────────
    # We must mask out the -1 labels so scikit-learn evaluates the true remaining structure
    valid_mask = final_labels != -1
    valid_features = cluster_features.values[valid_mask]
    valid_labels = final_labels[valid_mask]

    n_unique_final = len(np.unique(valid_labels))
    if n_unique_final > 1:
        sil_score = silhouette_score(valid_features, valid_labels)
        sil_samples = silhouette_samples(valid_features, valid_labels)
        cluster_silhouette = {
            int(lbl): sil_samples[valid_labels == lbl].mean()
            for lbl in np.unique(valid_labels)
        }
    else:
        sil_score = -1.0
        cluster_silhouette = {}

    return (cluster_dataset, fitted_cluster_model, sil_score,
            cluster_silhouette, cluster_label_map, train_dataset, merge_map)

In [16]:
def cluster_test_data(cluster_model, semester_name, merge_map=None):
    """
    Assign test-semester student-course instances to clusters using the
    fitted KMeans model.

    Each (student, course) row is assigned independently based on its
    course features — the same direct assignment used during training.

    Returns:
        dict: {cluster_label: {student_number: {course_code: grade, ...}}}
    """
    test_dataset = df[df.iloc[:, 3] == semester_name].copy()
    test_dataset.index = range(len(test_dataset))

    cluster_feature_cols = clustering_columns
    cluster_features_raw = test_dataset[cluster_feature_cols].copy()

    # Normalise with the same scaler implicitly by re-fitting on test slice.
    # (Since test features come from the same distribution, a fresh scaler
    # gives equivalent relative distances for cluster assignment.)
    sc_test = StandardScaler()
    cluster_features = pd.DataFrame(
        sc_test.fit_transform(cluster_features_raw),
        columns=cluster_feature_cols)

    cluster_labels = cluster_model.predict(cluster_features)
    semester_data  = get_semester_data(semester_name)

    cluster_dataset = {}
    for i in range(len(cluster_labels)):
        sn  = test_dataset.iloc[i, 0]
        cc  = test_dataset.iloc[i, 1]
        lbl = int(cluster_labels[i])

        if merge_map is not None:
            lbl = merge_map.get(lbl, lbl)

        cluster_dataset.setdefault(lbl, {})
        cluster_dataset[lbl].setdefault(sn, {})
        if sn in semester_data and cc in semester_data[sn]:
            cluster_dataset[lbl][sn][cc] = semester_data[sn][cc]

    return cluster_dataset

from scipy.spatial.distance import euclidean
import numpy as np

from scipy.spatial.distance import euclidean
import numpy as np

def get_semester_centroid_safe(semester_name, dataset_df, leaky_column_indices):
    """
    Calculates the macro-feature profile of a semester strictly using
    pre-semester (Day 1) knowledge to prevent data leakage.
    """
    # 1. Filter rows for this specific semester
    sem_data = dataset_df[dataset_df.iloc[:, 3] == semester_name]

    # 2. Extract ALL clustering features (Assuming they start at Col 4)
    all_features = sem_data.iloc[:, 4:]

    # 3. ─── AMPUTATE THE LEAKY FEATURES ──────────────────────────────
    # Drop the columns that represent actual grades, pass rates, or std-devs
    # of the current semester.
    safe_features = all_features.drop(all_features.columns[leaky_column_indices], axis=1).values

    # Failsafe for empty semesters
    if len(safe_features) == 0:
        num_safe_features = all_features.shape[1] - len(leaky_column_indices)
        return np.zeros(num_safe_features)

    # 4. Calculate the Centroid using ONLY the safe structural features
    semester_fingerprint = np.mean(safe_features, axis=0)

    return semester_fingerprint
def get_semester_centroid(semester_name, dataset_df):
    """
    Calculates the true macro-feature profile (centroid) of an entire semester.
    It extracts the exact same features K-Means uses (Credit, Subject, Grade Dist, etc.)
    and averages them to create a unique 'fingerprint' for the semester.
    """
    # 1. Filter the dataset to only include rows from this specific semester
    # (Assuming Column 3 is the Semester Name based on your sorting logic)
    sem_data = dataset_df[dataset_df.iloc[:, 3] == semester_name]

    # 2. Extract the clustering features (Everything from Column 4 onwards)
    features = sem_data.iloc[:, 4:].values

    # 3. Failsafe for empty semesters
    if len(features) == 0:
        num_features = dataset_df.shape[1] - 4
        return np.zeros(num_features)

    # 4. Calculate the Centroid (the mathematical average of all features)
    semester_fingerprint = np.mean(features, axis=0)

    return semester_fingerprint

def chronological_sort_key(semester_name):
    # Splits "2012 - Spring" into year=2012 and season="Spring"
    parts = semester_name.split(' - ')
    year = int(parts[0])
    season = parts[1].strip().lower()

    # Assign a chronological weight to the seasons
    season_weights = {
        'spring': 1,
        'summer': 2,
        'fall': 3
    }

    weight = season_weights.get(season, 4)

    # Returns a tuple like (2012, 1) so Python sorts by Year first, then Season
    return (year, weight)



In [17]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

def compute_overall_metrics(predictions, k_value='dynamic_k'):

    all_y_true = []
    all_y_pred = []

    if k_value not in predictions:
        print(f"Error: Target key '{k_value}' not found in predictions dictionary.")
        return None

    # Aggregate predictions across all tested semesters
    for sem_idx in predictions[k_value]:
        # Exclude 'cv_log' or other metadata keys if they were accidentally placed at the semester level
        if isinstance(predictions[k_value][sem_idx], dict) and 'y_true' in predictions[k_value][sem_idx]:
            all_y_true.extend(predictions[k_value][sem_idx]['y_true'])
            all_y_pred.extend(predictions[k_value][sem_idx]['y_pred'])

    # Calculate overall metrics
    if len(all_y_true) > 0:
        overall_rmse = np.sqrt(mean_squared_error(all_y_true, all_y_pred))
        overall_mae = mean_absolute_error(all_y_true, all_y_pred)
        total_n = len(all_y_true)

        print(f"--- Overall Metrics ({k_value}) ---")
        print(f"  RMSE: {overall_rmse:.4f}")
        print(f"  MAE:  {overall_mae:.4f}")
        print(f"  Total Predictions (N): {total_n:,}")

        return {
            'RMSE': overall_rmse,
            'MAE': overall_mae,
            'N': total_n
        }
    else:
        print("No predictions found to compute metrics.")
        return None

## Section 5: Main Prediction Function

Course-based adaptive per-cluster regression.

**Clustering unit:** Each student-course instance is assigned to a cluster
based on course-level features (credit, subject, year, aggregate grade stats).
A single student can appear in multiple clusters.

**Model selection:** 5-candidate holdout CV per cluster, same as student-based.


## Section 7: Analysis & Diagnostics

In [18]:
from scipy.spatial.distance import euclidean, cdist
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, RegressorMixin

def predict_regression_only(sim, threshold, k_range=range(10, 95, 5),
                            include_cf_candidate=False, test_all_models=False,
                            tolerance=0.85, global_min_silhouette=0.15, top_n_candidates=3,
                            model_selection_only=False):

    predictions = {}
    all_semesters = list(df.iloc[:, 3].unique())
    sorted_semesters = sorted(all_semesters, key=chronological_sort_key)

    print("Chronological Order:", sorted_semesters)

    predictions['dynamic_k'] = {}
    train_semester  = {}
    dynamic_k_log = {}

    def _is_invalid(val):
        if val is None: return True
        if isinstance(val, (int, float, np.number)):
            return np.isinf(val) or np.isnan(val)
        return True

    def _enforce_regression_winner(cv_log_dict, reg_models_dict, all_fitted_dict):
        for cid, winner_name in list(reg_models_dict.items()):
            if winner_name == 'CF' or winner_name == 'cf':
                best_m = None
                best_s = float('inf')
                for m_name, score in cv_log_dict.get(cid, {}).items():
                    if m_name.lower() == 'cf': continue
                    try:
                        v = float(score)
                        if not np.isinf(v) and not np.isnan(v) and v < best_s:
                            best_s = v
                            best_m = m_name
                    except: pass

                if best_m and cid in all_fitted_dict and best_m in all_fitted_dict[cid]:
                    reg_models_dict[cid] = all_fitted_dict[cid][best_m]
        return reg_models_dict

    # ─── BATCH OPTIMIZED ENSEMBLE EVALUATOR ──────────────────────────────────
    def _eval_ensemble_on_train(c_artifacts, r_artifacts, valid_train_sems):
        """
        Evaluates the ensemble using the standardized get_reg_train_data workflow.
        """
        (train_c_data, fitted_c_model, _, _, _, train_ds_reg, m_map) = c_artifacts
        r_mods, r_scaler, _, _ = r_artifacts

        # 1. Standard Clustering Scaler (centroids are based on this)
        sc_c = StandardScaler()
        sc_c.fit(train_ds_reg[clustering_columns])
        centroids = fitted_c_model.cluster_centers_

        # 2. CF Cache
        cf_cache = {lbl: recommendations.calculateSimilarItems(train_c_data[lbl])
                    for lbl, mod in r_mods.items() if mod == 'CF'}

        # 3. Determine Evaluation Data (Temporal or 20% split fallback)
        # Use get_reg_train_data to ensureCourse Titles/Strings are already removed

        # Check if we are in the first semester (Cold Start)
        if len(valid_train_sems) == 1:
            target_sem = valid_train_sems[0]
            # Get cleaned data via your standard helper
            #X_all, y_all = get_reg_train_data([s_name])

            # Split: 80% becomes "Pseudo-History", 20% becomes "Evaluation Target"
            #X_hist, X_eval, y_hist, y_eval = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

            # Use X_eval/y_eval for the RMSE calculation below
        else:
            # Standard temporal logic for later semesters
            target_sem = valid_train_sems[-2]

        X_eval, y_eval = get_reg_train_data([target_sem])

        # 4. Prepare Reg
        #
        # ression Predictions (Batch)
        # Ensure columns match what the regression models were trained on
        X_eval_aligned = X_eval.reindex(columns=r_scaler.feature_names_in_, fill_value=0.0)

        scaled_matrix = r_scaler.transform(X_eval_aligned)
        scaled_reg_matrix = np.nan_to_num(scaled_matrix, nan=0.0)  # Neutralize NaN features

        model_preds_cache = {}
        unique_models = {mod for mod in r_mods.values() if isinstance(mod, (BaseEstimator, RegressorMixin))}

        for mod in unique_models:
            raw_preds = mod.predict(scaled_reg_matrix)
            # Convert prediction indices back to numeric grades
            idx_arr = np.clip(np.round(raw_preds), 0, 12).astype(int)
            model_preds_cache[mod] = np.array([numerical_grades[le_reg.inverse_transform([idx])[0]] for idx in idx_arr])

        # centroids use 'clustering_columns'
        # 5. Prepare Clustering Weights
        # Instead of slicing X_eval, we map the student-course pairs back to the original clustering features
        # meta_info contains (Student Number, Course Code) for the evaluation rows
        meta_info = df.loc[X_eval.index, [df.columns[0], df.columns[1]]]

        # Extract clustering features directly from the main 'df' using the index of the evaluation rows
        X_clustering = df.loc[X_eval.index, clustering_columns]

        # Ensure no NaNs exist (standard guard for new environments)
        X_clustering = X_clustering.apply(pd.to_numeric, errors='coerce').fillna(0)

        # Now proceed with scaling and distance calculation
        scaled_clustering_matrix = sc_c.transform(X_clustering)
        all_dists = cdist(scaled_clustering_matrix, centroids, metric='euclidean')

        # Calculate Inverse Distance Weighting
        inv_sq = 1.0 / (all_dists**2 + 1e-8)
        weights_matrix = inv_sq / inv_sq.sum(axis=1, keepdims=True)


        # 6. Final Ensemble RMSE Calculation
        y_true, y_pred = [], []

        # meta_info is a DataFrame (supports .iloc)
        # df.columns[0] = Student Number, df.columns[1] = Course Code
        meta_info = df.loc[X_eval.index, [df.columns[0], df.columns[1]]]

        for i in range(len(X_eval)):
            # Access metadata from DataFrame
            sn, cc = meta_info.iloc[i].values
            weights = weights_matrix[i]

            row_w_pred, row_v_wgt = 0.0, 0.0

            for cluster_idx, w in enumerate(weights):
                if w < 0.01: continue

                orig_lbl = cluster_idx
                eval_lbl = m_map.get(orig_lbl, orig_lbl)
                model_obj = r_mods.get(eval_lbl)

                pred = None
                if model_obj == 'CF':
                    c_tr = train_c_data.get(eval_lbl, {})
                    try:
                        res = recommendations.getRecommendedItems(c_tr, cf_cache.get(eval_lbl), sn)
                        for rg, rc in res:
                            if rc == cc: pred = rg; break
                    except: pass
                elif model_obj in model_preds_cache:
                    pred = model_preds_cache[model_obj][i]

                if pred is not None:
                    row_w_pred += w * pred
                    row_v_wgt += w

            if row_v_wgt > 0:
                # FIX: Use [i] instead of .iloc[i] because y_eval is a NumPy array
                y_true.append(y_eval[i])
                y_pred.append(row_w_pred / row_v_wgt)

        if len(y_true) == 0:
            print("DEBUG: WARNING: Empty y_true, return RMSE 2.5")
            return 2.5
        else:
            return np.sqrt(mean_squared_error(y_true, y_pred))

    #104
    for sem_idx in range(1, len(sorted_semesters)): #
        print(f"\n---> Optimizing k for Semester {sem_idx} ({sorted_semesters[sem_idx]})...")
        predictions['dynamic_k'].setdefault(
            str(sem_idx), {'y_true': [], 'y_pred': [], 'y_preds': {}, 'sources': [], 'cv_log': {}})


        anchor_sem_name = sorted_semesters[sem_idx - 1]
        valid_training_semesters = [anchor_sem_name]
        for back_idx in range(sem_idx - 2, -1, -1):
            historical_sem_name = sorted_semesters[back_idx]
            valid_training_semesters.insert(0, historical_sem_name)
        training_semesters_name = valid_training_semesters

        # STRICT TIMELINE: Only use semesters that occurred BEFORE the test semester
        #training_semesters_name = sorted_semesters[:sem_idx]

        train_semester = {}
        print("\nTraining Semesters: ")
        for s_name in training_semesters_name:
            print(s_name)
            sem_data = get_semester_data(s_name)
            for student in sem_data:
                filtered_grades = {cc: grade for cc, grade in sem_data[student].items() if grade >= 0.0}
                if not filtered_grades:
                    continue # Skip student if they only had Fs this semester
                # ─────────────────────────────────────────────────────────

                if student in train_semester:
                    train_semester[student].update(filtered_grades)
                else:
                    train_semester[student] = filtered_grades.copy()

        candidates = []
        max_sil_score = -1.0
        last_merge_signature = None

        print("     [Stage 1] Calculating Silhouette Scores...")
        test_range = [k for k in k_range if k > 1]

        for num_clusters in test_range:
            with warnings.catch_warnings():
                warnings.filterwarnings("ignore", category=ConvergenceWarning)
                warnings.filterwarnings("ignore", message=".*Number of distinct clusters.*")
                artifacts = fit_cluster(training_semesters_name, num_clusters, train_semester, KMeans, threshold)


            sil_score = artifacts[2]
            current_sil = sil_score if not np.isnan(sil_score) else -1.0
            candidates.append({'k': num_clusters, 'sil': current_sil, 'artifacts': artifacts})
            if current_sil > max_sil_score: max_sil_score = current_sil

            fitted_kmeans = artifacts[1]
            actual_k_found = len(set(fitted_kmeans.labels_))
            if actual_k_found < num_clusters: break

            total_instances = sum(sum(len(c) for c in s.values()) for s in artifacts[0].values())
            if (total_instances / threshold) < 2:
                print(f"     >> Impossible to form more than one valid cluster. Early stopping.")
                break

            current_merge_signature = tuple(sorted([sum(len(c) for c in s.values()) for s in artifacts[0].values()]))

            # ─── FUZZY ASYMPTOTE EARLY STOPPING ───
            if last_merge_signature is not None and len(current_merge_signature) == len(last_merge_signature):
                size_diff = sum(abs(a - b) for a, b in zip(current_merge_signature, last_merge_signature))

                # If the shifting boundaries represent less than 5% of total data, stop the sweep.
                # (Using 5% here ensures we don't stop prematurely if a genuine new cluster forms)
                if size_diff < (total_instances * 0.005):
                    print(f"     >> Fuzzy Structural Asymptote Reached: Only {size_diff} instances shifted. Early stopping K-sweep.")
                    break

            last_merge_signature = current_merge_signature

        best_k = None
        best_sil_score = -1.0
        final_clustering_artifacts = None
        final_regression_artifacts = None
        winning_cv_std = float('inf')
        winning_ensemble_rmse = float('inf')

        if max_sil_score < global_min_silhouette:
            print(f"     >> Data lacks strong structure (Max Sil {max_sil_score:.3f} < {global_min_silhouette}). Defaulting to k=1.")
            best_k = 1
            best_sil_score = -1.0
            final_clustering_artifacts = fit_cluster(training_semesters_name, 1, train_semester, KMeans, threshold)
            _supply_cf = include_cf_candidate or test_all_models
            reg_models, reg_scaler, cv_log, all_fitted = fit_regression_models_per_cluster(
                training_semesters_name, final_clustering_artifacts[4], final_clustering_artifacts[5],
                tuned_hyperparams_reg, threshold, cluster_raw_data=final_clustering_artifacts[0] if _supply_cf else None,
                sim_func=sim if _supply_cf else None, test_all_models=test_all_models, train_semester_global=train_semester,
                model_selection_only=model_selection_only)

            if model_selection_only:
                continue

            if not include_cf_candidate: reg_models = _enforce_regression_winner(cv_log, reg_models, all_fitted)
            final_regression_artifacts = (reg_models, reg_scaler, cv_log, all_fitted)
            winning_ensemble_rmse = _eval_ensemble_on_train(final_clustering_artifacts, final_regression_artifacts, training_semesters_name)

        else:
            unique_valid_candidates = []
            seen_signatures = []
            valid_clustered_candidates = [c for c in candidates if len(c['artifacts'][0]) > 1]

            if len(valid_clustered_candidates) == 0: shortlist = []
            else:
                true_max_sil = max(c['sil'] for c in valid_clustered_candidates)
                target_threshold = true_max_sil * tolerance if true_max_sil > 0 else true_max_sil
                valid_clustered_candidates.sort(key=lambda x: x['sil'], reverse=True)
                total_instances = sum(sum(len(c) for c in s.values()) for s in valid_clustered_candidates[0]['artifacts'][0].values())
                FUZZY_TOLERANCE = total_instances * 0.10

                for c in valid_clustered_candidates:
                    if c['sil'] >= target_threshold:
                        cluster_label_map = c['artifacts'][4]
                        cluster_sizes = {}
                        for lbl in cluster_label_map.values(): cluster_sizes[lbl] = cluster_sizes.get(lbl, 0) + 1

                        signature = tuple(sorted(cluster_sizes.values()))
                        is_duplicate = False

                        for seen_sig in seen_signatures:
                            if len(signature) == len(seen_sig):
                                size_diff = sum(abs(a - b) for a, b in zip(signature, seen_sig))
                                if size_diff < FUZZY_TOLERANCE:
                                    is_duplicate = True
                                    break

                        if not is_duplicate:
                            seen_signatures.append(signature)
                            unique_valid_candidates.append(c)

                shortlist = unique_valid_candidates[:top_n_candidates]

            if model_selection_only:
                continue

            if len(shortlist) == 0:
                print(f"     [Stage 2 & 3 Bypassed] All valid candidates collapsed to a single global cluster.")
                best_k = 1
                winning_cv_std = 0.0
                artifacts_k1 = fit_cluster(training_semesters_name, 1, train_semester, KMeans, threshold)
                final_clustering_artifacts = artifacts_k1
                _supply_cf = include_cf_candidate or test_all_models
                reg_models_opt, reg_scaler_opt, cv_log_opt, all_fitted_opt = fit_regression_models_per_cluster(
                    training_semesters_name, final_clustering_artifacts[4], final_clustering_artifacts[5],
                    tuned_hyperparams_reg, threshold, cluster_raw_data=final_clustering_artifacts[0] if _supply_cf else None,
                    sim_func=sim if _supply_cf else None, test_all_models=test_all_models, train_semester_global=train_semester,
                    model_selection_only=model_selection_only)

                if not include_cf_candidate: reg_models_opt = _enforce_regression_winner(cv_log_opt, reg_models_opt, all_fitted_opt)
                final_regression_artifacts = (reg_models_opt, reg_scaler_opt, cv_log_opt, all_fitted_opt)
                winning_ensemble_rmse = _eval_ensemble_on_train(final_clustering_artifacts, final_regression_artifacts, training_semesters_name)

            else:
                print(f"     [Stage 2] Evaluating Soft-Ensemble Train-RMSE for Top {len(shortlist)} UNIQUE candidates...")
                winning_ensemble_rmse = float('inf')

                for candidate in shortlist:
                    cand_k = candidate['k']
                    cand_sil = candidate['sil']
                    cand_artifacts = candidate['artifacts']
                    (train_cluster_data, fitted_cluster_model, sil_score, cluster_silhouette, cluster_label_map, train_dataset_for_reg, merge_map) = cand_artifacts

                    _supply_cf = include_cf_candidate or test_all_models
                    reg_models, reg_scaler, cv_log, all_fitted = fit_regression_models_per_cluster(
                        training_semesters_name, cluster_label_map, train_dataset_for_reg, tuned_hyperparams_reg, threshold,
                        cluster_raw_data=train_cluster_data if _supply_cf else None, sim_func=sim if _supply_cf else None,
                        test_all_models=test_all_models, train_semester_global=train_semester,
                        model_selection_only=model_selection_only)

                    if not include_cf_candidate: reg_models = _enforce_regression_winner(cv_log, reg_models, all_fitted)
                    cand_reg_artifacts = (reg_models, reg_scaler, cv_log, all_fitted)

                    cluster_stds = []
                    for cluster_id, models in cv_log.items():
                        cluster_model_scores = []
                        for model_name, score in models.items():
                            if model_name.lower() == 'cf' and not include_cf_candidate: continue
                            try:
                                val = float(score)
                                if not np.isinf(val) and not np.isnan(val): cluster_model_scores.append(val)
                            except: pass
                        if len(cluster_model_scores) > 0: cluster_stds.append(np.std(cluster_model_scores))

                    current_cv_std = np.mean(cluster_stds) if len(cluster_stds) > 0 else float('inf')
                    current_ensemble_rmse = _eval_ensemble_on_train(cand_artifacts, cand_reg_artifacts, training_semesters_name)

                    print(f"       Tested k={cand_k:02d} | Sil: {cand_sil:.3f} | Reg CV-Std: {current_cv_std:.4f} | Ens Train-HEM: {current_ensemble_rmse:.4f}")

                    if current_ensemble_rmse < winning_ensemble_rmse:
                        winning_ensemble_rmse = current_ensemble_rmse
                        winning_cv_std = current_cv_std
                        best_k = cand_k
                        best_sil_score = cand_sil
                        final_clustering_artifacts = cand_artifacts
                        final_regression_artifacts = cand_reg_artifacts

                print(f"     >> Stage 2 SUCCESS: Selected candidate k={best_k} (Lowest Ens Train-HEM={winning_ensemble_rmse:.4f})")

            final_cluster_count = len(final_clustering_artifacts[0])

            if final_cluster_count <= 1:
                print(f"     [Stage 3] Bypassed: Candidate k={best_k} collapsed into a single global cluster.")
                best_k = 1
                best_sil_score = -1.0
                winning_cv_std = float('inf')
            else:
                if False:
                    print(f"     [Stage 3] Supervised Sanity Check (Soft-Ensemble): k={best_k} vs k=1")
                    artifacts_k1 = fit_cluster(training_semesters_name, 1, train_semester, KMeans, threshold)
                    _supply_cf = include_cf_candidate or test_all_models
                    reg_models_1, reg_scaler_1, cv_log_1, all_fitted_1 = fit_regression_models_per_cluster(
                        training_semesters_name, artifacts_k1[4], artifacts_k1[5], tuned_hyperparams_reg, threshold,
                        cluster_raw_data=artifacts_k1[0] if _supply_cf else None, sim_func=sim if _supply_cf else None,
                        test_all_models=test_all_models, train_semester_global=train_semester,
                        model_selection_only=model_selection_only)

                    if not include_cf_candidate: reg_models_1 = _enforce_regression_winner(cv_log_1, reg_models_1, all_fitted_1)
                    final_regression_artifacts_k1 = (reg_models_1, reg_scaler_1, cv_log_1, all_fitted_1)

                    print("       -> Simulating Inverse Distance Weighting across training history for k=1...")
                    #rmse_k1_ens = _eval_ensemble_on_train(artifacts_k1, final_regression_artifacts_k1, training_semesters_name)
                    rmse_k1_ens = 1000000
                    rmse_k_opt_ens = winning_ensemble_rmse

                    print(f"       -> k=1 Global Ensemble Train-RMSE:   {rmse_k1_ens:.4f}")
                    print(f"       -> k={best_k} Weighted Ensemble Train-RMSE: {rmse_k_opt_ens:.4f}")

                    K1_TOLERANCE = 1.00
                    if rmse_k1_ens <= (rmse_k_opt_ens * K1_TOLERANCE):
                        print(f"     >> k=1 Outperformed or tied k={best_k} (with {K1_TOLERANCE}% penalty). Falling back to Global Model.")
                        best_k = 1
                        best_sil_score = -1.0
                        winning_cv_std = float('inf')
                        winning_ensemble_rmse = rmse_k1_ens
                        final_clustering_artifacts = artifacts_k1
                        final_regression_artifacts = final_regression_artifacts_k1
                    else:
                        print(f"     >> k={best_k} Ensemble Error is significantly lower. Keeping Clustered Model.")

        dynamic_k_log[str(sem_idx)] = {
            'best_k': best_k,
            'sil_score': best_sil_score,
            'cv_std': winning_cv_std if best_k > 1 else 'N/A'
        }

        # ─── EXTRACTION & INFERENCE (FINAL TEST LOOP) ─────────────────────────
        (train_cluster_data, fitted_cluster_model, sil_score, cluster_silhouette,
         cluster_label_map, train_dataset_for_reg, merge_map) = final_clustering_artifacts

        actual_final_clusters = len(train_cluster_data)
        reg_models, reg_scaler, cv_log, all_fitted = final_regression_artifacts
        predictions['dynamic_k'][str(sem_idx)]['best_k'] = best_k
        predictions['dynamic_k'][str(sem_idx)]['sil_score'] = sil_score
        predictions['dynamic_k'][str(sem_idx)]['cluster_silhouette'] = cluster_silhouette
        predictions['dynamic_k'][str(sem_idx)]['cv_log'] = cv_log
        test_semester_name = sorted_semesters[sem_idx]
        test_data_full = get_semester_data(test_semester_name)

        sc_cluster = StandardScaler()
        sc_cluster.fit(train_dataset_for_reg[clustering_columns])

        cf_sims_cache, all_model_cf_sims = {}, {}
        for lbl in train_cluster_data:
            if reg_models.get(lbl) == 'CF':
                cf_sims_cache[lbl] = recommendations.calculateSimilarItems(train_cluster_data[lbl])
            if test_all_models and 'CF' in all_fitted.get(lbl, {}):
                all_model_cf_sims[lbl] = recommendations.calculateSimilarItems(train_cluster_data[lbl])

        warnings.filterwarnings("ignore", message="X does not have valid feature names, but")

        # ── OPTIMIZATION 1: Vectorized Test Cache ──
        test_sem_df = df[df.iloc[:, 3] == test_semester_name]
        unique_test_courses = test_sem_df.drop_duplicates(subset=[df.columns[1]])
        test_course_weight_cache = {}

        if not unique_test_courses.empty:
            raw_test_feats = unique_test_courses[clustering_columns].values
            scaled_test_feats = sc_cluster.transform(raw_test_feats)
            all_test_dists = cdist(scaled_test_feats, fitted_cluster_model.cluster_centers_, metric='euclidean')
            for idx, cc in enumerate(unique_test_courses.iloc[:, 1].values):
                dists = all_test_dists[idx]
                inv_sq = 1.0 / ((dists**2) + 1e-8)
                test_course_weight_cache[cc] = inv_sq / np.sum(inv_sq)

        # ── OPTIMIZATION 2: MATRIX PRE-PREDICTIONS FOR TEST ──
        test_sem_reg_rows = df_reg[df_reg['Semester'] == test_semester_name].copy()
        test_model_preds_cache = {}
        if len(test_sem_reg_rows) > 0 and reg_scaler is not None:
            test_stu_col = test_sem_reg_rows['Student Number'].values
            test_cc_col = test_sem_reg_rows['Course Code'].values
            for c in ['Semester', 'Letter Grade', 'Student Number', 'Course Code']:
                if c in test_sem_reg_rows.columns: test_sem_reg_rows = test_sem_reg_rows.drop(c, axis=1)

            missing_cols = set(reg_scaler.feature_names_in_) - set(test_sem_reg_rows.columns)
            for c in missing_cols: test_sem_reg_rows[c] = 0.0
            test_sem_reg_rows = test_sem_reg_rows[reg_scaler.feature_names_in_]

            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    scaled_test_matrix = reg_scaler.transform(test_sem_reg_rows)

                unique_test_models = set(m for m in reg_models.values() if m != 'CF' and m is not None)
                if test_all_models:
                    for cluster_models in all_fitted.values():
                        # Added 'and m is not None' for extra safety
                        unique_test_models.update([m for m in cluster_models.values() if m != 'CF' and m is not None])

                for model in unique_test_models:
                    # NEW: Inner try-catch isolates unstable candidate models
                    try:
                        with warnings.catch_warnings():
                            warnings.simplefilter("ignore")
                            raw_preds = model.predict(scaled_test_matrix)
                        idx_arr = np.clip(np.round(raw_preds), 0, 12).astype(int)
                        converted_grades = np.array([numerical_grades[le_reg.inverse_transform([idx])[0]] for idx in idx_arr])

                        m_dict = {}
                        for i in range(len(test_stu_col)):
                            m_dict[(test_stu_col[i], test_cc_col[i])] = converted_grades[i]
                        test_model_preds_cache[model] = m_dict
                    except Exception:
                        pass # Skip only this specific model
            except Exception:
                pass # Skip if the scaler fails
        # ─────────────────────────────────────────────────────

        for student in test_data_full:
            for course_code, true_grade in test_data_full[student].items():
                final_pred, source_tag = None, ""

                if course_code not in test_course_weight_cache:
                    try:
                        final_pred = np.mean(list(train_semester.get(student, {}).values()))
                        source_tag = "Global_GPA_Fallback_Missing_Course"
                    except:
                        final_pred, source_tag = 2.0, "Hard_Fallback"
                else:
                    weights = test_course_weight_cache[course_code]
                    final_weighted_pred, valid_weight_sum = 0.0, 0.0

                    for original_lbl, weight in enumerate(weights):
                        if weight < 0.01: continue

                        effective_label = merge_map.get(original_lbl, original_lbl)
                        cluster_pred = None
                        model_obj = reg_models.get(effective_label)

                        if model_obj == 'CF':
                            cluster_train = train_cluster_data.get(effective_label, {})
                            gpa_local = {s: get_avg_gpa(cluster_train, s) for s in cluster_train}
                            recommended = {}
                            cache = cf_sims_cache.get(effective_label)
                            try:
                                for rec_grade, rec_course in recommendations.getRecommendedItems(cluster_train, cache, student): recommended[rec_course] = rec_grade
                                for rec_grade, rec_course in recommendations.getRecommendations(cluster_train, student, sim, dgpa=True, gpa=gpa_local, delta=0.7):
                                    if rec_course not in recommended: recommended[rec_course] = rec_grade
                            except: pass
                            if course_code in recommended: cluster_pred = recommended[course_code]
                            else:
                                try: cluster_pred = gpa_local[student]
                                except: pass

                        elif model_obj is not None:
                            # ── O(1) Instant Cache Lookup ──
                            if model_obj in test_model_preds_cache:
                                cluster_pred = test_model_preds_cache[model_obj].get((student, course_code))

                        if cluster_pred is not None:
                            final_weighted_pred += weight * cluster_pred
                            valid_weight_sum += weight

                    if valid_weight_sum > 0:
                        final_pred = final_weighted_pred / valid_weight_sum
                        source_tag = "Weighted_Ensemble"
                    else:
                        # 1. Retrieve the student's history first
                        stu_history = list(train_semester.get(student, {}).values())

                        # 2. Check if the list actually has data before calling np.mean
                        if len(stu_history) > 0:
                            final_pred = np.mean(stu_history)
                            source_tag = "Global_GPA_Fallback"
                        else:
                            # This handles students with NO history without triggering a warning
                            final_pred = 2.0
                            source_tag = "Hard_Fallback"


                predictions['dynamic_k'][str(sem_idx)]['y_true'].append(true_grade)
                predictions['dynamic_k'][str(sem_idx)]['y_pred'].append(final_pred)
                predictions['dynamic_k'][str(sem_idx)]['sources'].append(source_tag)

                if test_all_models:
                    sem_preds = predictions['dynamic_k'][str(sem_idx)]['y_preds']
                    candidate_models = set()
                    for c_models in all_fitted.values(): candidate_models.update(c_models.keys())
                    if include_cf_candidate: candidate_models.add('CF')

                    for m_name in candidate_models:
                        m_weighted_pred, m_valid_weight = 0.0, 0.0

                        if course_code in test_course_weight_cache:
                            for original_lbl, weight in enumerate(weights):
                                if weight < 0.01: continue

                                effective_label = merge_map.get(original_lbl, original_lbl)
                                m_cluster_pred = None
                                fitted_for_cluster = all_fitted.get(effective_label, {})

                                if m_name == 'CF' and m_name in fitted_for_cluster:
                                    cluster_train = train_cluster_data.get(effective_label, {})
                                    gpa_local_all = {s: get_avg_gpa(cluster_train, s) for s in cluster_train}
                                    rec_all = {}
                                    cache_all = all_model_cf_sims.get(effective_label)
                                    try:
                                        for rg, rc in recommendations.getRecommendedItems(cluster_train, cache_all, student): rec_all[rc] = rg
                                        for rg, rc in recommendations.getRecommendations(cluster_train, student, sim, dgpa=True, gpa=gpa_local_all, delta=0.7):
                                            if rc not in rec_all: rec_all[rc] = rg
                                    except: pass
                                    if course_code in rec_all: m_cluster_pred = rec_all[course_code]
                                    else:
                                        try: m_cluster_pred = gpa_local_all[student]
                                        except: pass

                                elif m_name in fitted_for_cluster:
                                    m_obj = fitted_for_cluster[m_name]
                                    # ── O(1) Instant Cache Lookup ──
                                    if m_obj in test_model_preds_cache:
                                        m_cluster_pred = test_model_preds_cache[m_obj].get((student, course_code))

                                if m_cluster_pred is not None:
                                    m_weighted_pred += weight * m_cluster_pred
                                    m_valid_weight += weight

                        if m_valid_weight > 0: final_m_pred = m_weighted_pred / m_valid_weight
                        else:
                            stu_history = list(train_semester.get(student, {}).values())

                            # 2. Check if the list actually has data before calling np.mean
                            if len(stu_history) > 0:
                                final_m_pred = np.mean(stu_history)
                                source_tag = "Global_GPA_Fallback"
                            else:
                                # This handles students with NO history without triggering a warning
                                final_m_pred = 2.0
                                source_tag = "Hard_Fallback"

                        sem_preds.setdefault(m_name, {'y_true': [], 'y_pred': []})
                        sem_preds[m_name]['y_true'].append(true_grade)
                        sem_preds[m_name]['y_pred'].append(final_m_pred)

        sem_y_true = predictions['dynamic_k'][str(sem_idx)]['y_true']
        sem_y_pred = predictions['dynamic_k'][str(sem_idx)]['y_pred']

        sem_rmse, sem_mae = np.nan, np.nan
        if len(sem_y_true) > 0:
            sem_rmse = np.sqrt(mean_squared_error(sem_y_true, sem_y_pred))
            sem_mae  = mean_absolute_error(sem_y_true, sem_y_pred)
            print(f"  Sem {sem_idx} Done. N: {len(sem_y_true):,} | Test RMSE: {sem_rmse:.4f} | Test MAE: {sem_mae:.4f}")

        # --- NEW: Model Selection vs. Actual Performance Comparison ---
        if test_all_models:
            print(f"\n[Model Selection vs. Actual Performance - Semester {sem_idx}]")
            print(f"{'Model Name':<15} | {'Val RMSE (Avg)':<15} | {'Test RMSE':<10}")
            print("-" * 45)

            # 1. Get the set of all models that were available in the clusters
            all_candidate_names = set()
            for c_id in cv_log:
                all_candidate_names.update(cv_log[c_id]['all_scores'].keys())

            for m_name in sorted(all_candidate_names):
                # 2. Compute Average Validation RMSE across all clusters for this model
                val_scores = []
                for c_id in cv_log:
                    score = cv_log[c_id]['all_scores'].get(m_name)
                    if score is not None and not np.isinf(score):
                        val_scores.append(score)

                avg_val_rmse = np.mean(val_scores) if val_scores else np.nan

                # 3. Get Actual Test RMSE from the predictions stored in y_preds
                m_test_rmse = np.nan
                if m_name in predictions['dynamic_k'][str(sem_idx)]['y_preds']:
                    m_data = predictions['dynamic_k'][str(sem_idx)]['y_preds'][m_name]
                    if len(m_data['y_true']) > 0:
                        m_test_rmse = np.sqrt(mean_squared_error(m_data['y_true'], m_data['y_pred']))

                print(f"{m_name:<15} | {avg_val_rmse:<15.4f} | {m_test_rmse:<10.4f}")
            print("-" * 45)

        dynamic_k_log[str(sem_idx)] = {
            'semester': test_semester_name,
            'req_k': best_k,
            'final_k': actual_final_clusters,
            'test_rmse': sem_rmse,
            'test_mae': sem_mae,
            'sil_score': best_sil_score,
            'train_ens_rmse': winning_ensemble_rmse
        }

    print("\n" + "="*95)
    print("--- FINAL DYNAMIC K SELECTION & ENSEMBLE PERFORMANCE LOG ---")
    print(f"{'Semester':<15} | {'Req K':<6} | {'Final K':<7} | {'Train Ens-RMSE':<14} | {'Test RMSE':<10} | {'Test MAE':<9} | {'Sil Score'}")
    print("-" * 95)
    for s, log in dynamic_k_log.items():
        print(f"{log['semester']:<15} | {log['req_k']:<6} | {log['final_k']:<7} | {log['train_ens_rmse']:<14.4f} | {log['test_rmse']:<10.4f} | {log['test_mae']:<9.4f} | {log['sil_score']:.3f}")
    print("="*95 + "\n")

    return predictions, dynamic_k_log

In [19]:
K_1 = False  # Last training semester is the validation
model_sel_only = False
#101

if K_1:
    k_values = list(range(1,2,1))
    reg_predictions_k10, dynamic_k_log = predict_regression_only(
        k_range=k_values,
        sim=recommendations.sim_distance,
        include_cf_candidate=True,
        tolerance = 0.05,
        global_min_silhouette=0.05,
        test_all_models=True,
        model_selection_only=model_sel_only,
        threshold = 590
    )

    if not model_sel_only:
        #file_name = 'adaptive_regression_results_course_based_KMeans.c150.TemporalModelSelectionArithmeticMean.json'
        file_name = 'adaptive_regression_results_course_based_KMeans.k1.ALL.NarrowFit.RMSE.json'
        # ── Save results ──────────────────────────────────────────────────────────
        with open(file_name, 'w') as fw:
            def to_serializable(obj):
                if isinstance(obj, dict):
                    return {k: to_serializable(v) for k, v in obj.items()}
                elif isinstance(obj, list):
                    return [to_serializable(i) for i in obj]
                elif isinstance(obj, (np.integer,)):
                    return int(obj)
                elif isinstance(obj, (np.floating,)):
                    return float(obj)
                elif isinstance(obj, np.ndarray):
                    return obj.tolist()
                return obj
            json.dump(to_serializable(reg_predictions_k10), fw)
        print("Results saved.")
        overall_results = compute_overall_metrics(reg_predictions_k10)
        print(overall_results)

In [20]:
K_ADAPTIVE = True  #Last training semester is the validation
model_sel_only = False
#103

if K_ADAPTIVE:
    k_values = list(range(1,101,1))
    reg_predictions_k10, dynamic_k_log = predict_regression_only(
        k_range=k_values,
        sim=recommendations.sim_distance,
        include_cf_candidate=True,
        tolerance = 0.05,
        global_min_silhouette=0.05,
        test_all_models=False,
        model_selection_only=model_sel_only,
        threshold = 300 #200
    )

    if not model_sel_only:
        #file_name = 'adaptive_regression_results_course_based_KMeans.c150.TemporalModelSelectionArithmeticMean.json'
        file_name = 'adaptive_regression_results_course_based_KMeans.c300.NarrowFit.RMSE.json'
        # ── Save results ──────────────────────────────────────────────────────────
        with open(file_name, 'w') as fw:
            def to_serializable(obj):
                if isinstance(obj, dict):
                    return {k: to_serializable(v) for k, v in obj.items()}
                elif isinstance(obj, list):
                    return [to_serializable(i) for i in obj]
                elif isinstance(obj, (np.integer,)):
                    return int(obj)
                elif isinstance(obj, (np.floating,)):
                    return float(obj)
                elif isinstance(obj, np.ndarray):
                    return obj.tolist()
                return obj
            json.dump(to_serializable(reg_predictions_k10), fw)
        print("Results saved.")
        overall_results = compute_overall_metrics(reg_predictions_k10)
        print(overall_results)

Chronological Order: ['2011 - Spring', '2011 - Fall', '2012 - Spring', '2012 - Fall', '2013 - Spring', '2013 - Fall', '2014 - Spring', '2014 - Fall']

---> Optimizing k for Semester 1 (2011 - Fall)...

Training Semesters: 
2011 - Spring
     [Stage 1] Calculating Silhouette Scores...
  After merging 2: 2 clusters, instance counts: [805, 1005]
  After merging 3: 2 clusters, instance counts: [805, 1005]
     >> Fuzzy Structural Asymptote Reached: Only 0 instances shifted. Early stopping K-sweep.
     [Stage 2] Evaluating Soft-Ensemble Train-RMSE for Top 1 UNIQUE candidates...
Cluster: 1
Validation target: 2011 - Spring
80 - 20 case...
  DEBUG: Alternating sems: ['2011 - Spring']
  DEBUG: Training rows: 1005
  Cluster 1: winner = Bagging (n=1005, AvgRMSE=1.7310)
Cluster: 0
Validation target: 2011 - Spring
80 - 20 case...
  DEBUG: Alternating sems: ['2011 - Spring']
  DEBUG: Training rows: 805
  Cluster 0: winner = Bagging (n=805, AvgRMSE=1.6030)
       Tested k=02 | Sil: 0.245 | Reg CV-St

In [71]:
THRESHOLD_SWEEP = True
BEST_K = 10
#102
if THRESHOLD_SWEEP:
    #threshold_values = list(range(4000, 6001, 200))
    threshold_values = [300, 590]
    #threshold_values  = [200, 400, 500, 700, 800, 900, 1000, 2000]
    threshold_results = {}
    k_values = list(range(1,101,1))

    for tau in threshold_values:
        THRESHOLD = tau
        print(f"\n{'='*60}")
        print(f"THRESHOLD = {tau}")
        print(f"{'='*60}")

        reg_predictions_k10, dynamic_k_log = predict_regression_only(
            k_range=k_values,
            sim=recommendations.sim_distance,
            include_cf_candidate=False,
            tolerance = 0.05,
            global_min_silhouette=0.05,
            test_all_models=False,
            model_selection_only=False,
            threshold = tau
        )


        #file_name = 'adaptive_regression_results_course_based_KMeans.c150.TemporalModelSelectionArithmeticMean.json'
        file_name = f'adaptive_regression_results_course_based_KMeans.c{tau}.NarrowFit.RMSE.json'
        # ── Save results ──────────────────────────────────────────────────────────
        with open(file_name, 'w') as fw:
            def to_serializable(obj):
                if isinstance(obj, dict):
                    return {k: to_serializable(v) for k, v in obj.items()}
                elif isinstance(obj, list):
                    return [to_serializable(i) for i in obj]
                elif isinstance(obj, (np.integer,)):
                    return int(obj)
                elif isinstance(obj, (np.floating,)):
                    return float(obj)
                elif isinstance(obj, np.ndarray):
                    return obj.tolist()
                return obj
            json.dump(to_serializable(reg_predictions_k10), fw)
        print("Results saved.")
        overall_results = compute_overall_metrics(reg_predictions_k10)
        print(overall_results)
        threshold_results[tau] = {'rmse': overall_results['RMSE'], 'mae': overall_results['MAE'], 'predictions': reg_predictions_k10}

    print("\n\nTHRESHOLD SWEEP SUMMARY (k=10)")
    print(f"{'Threshold':>12} {'RMSE':>8} {'MAE':>8}")
    print("-" * 32)
    for tau in threshold_values:
        r = threshold_results[tau]
        marker = " <-- current" if tau == 590 else ""
        print(f"{tau:>12} {r['rmse']:>8.4f} {r['mae']:>8.4f}{marker}")


THRESHOLD = 300
Chronological Order: ['2011 - Spring', '2011 - Fall', '2012 - Spring', '2012 - Fall', '2013 - Spring', '2013 - Fall', '2014 - Spring', '2014 - Fall']

---> Optimizing k for Semester 1 (2011 - Fall)...

Training Semesters: 
2011 - Spring
     [Stage 1] Calculating Silhouette Scores...
  After merging 2: 2 clusters, instance counts: [805, 1005]
  After merging 3: 2 clusters, instance counts: [805, 1005]
     >> Fuzzy Structural Asymptote Reached: Only 0 instances shifted. Early stopping K-sweep.
     [Stage 2] Evaluating Soft-Ensemble Train-RMSE for Top 1 UNIQUE candidates...
Cluster: 1
Validation target: 2011 - Spring
80 - 20 case...
  DEBUG: Alternating sems: ['2011 - Spring']
  DEBUG: Training rows: 1005
  Cluster 1: winner = Bagging (n=1005, AvgRMSE=1.7310)
Cluster: 0
Validation target: 2011 - Spring
80 - 20 case...
  DEBUG: Alternating sems: ['2011 - Spring']
  DEBUG: Training rows: 805
  Cluster 0: winner = Bagging (n=805, AvgRMSE=1.6030)
       Tested k=02 | Sil: 